# OfflineMedia — Long-Form Video Generator

Generates videos of **any length** (tested up to 15 min) by chaining Wan2GP clips and stitching them with FFmpeg.

**Run cells top to bottom.** On a free T4 GPU, each 8-second clip takes ~5 min. A 15-min video (~112 clips) needs ~9 hours — use **Colab Pro** for sessions that long.

> **Tip:** Enable Google Drive in Cell 2 to keep models and outputs across restarts.

In [ ]:
# Cell 1 — GPU check
import subprocess

result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
    capture_output=True, text=True,
)
if result.returncode != 0:
    raise RuntimeError('No GPU found. Runtime -> Change runtime type -> GPU')
print('GPU:', result.stdout.strip())

In [ ]:
# Cell 2 — Configure storage
# Set USE_GOOGLE_DRIVE = True to keep models and outputs across Colab restarts.
USE_GOOGLE_DRIVE = False
DRIVE_PATH = 'MyDrive/Wan2GP-data'

In [ ]:
# Cell 3 — Clone / update OfflineMedia repo
import subprocess, sys
from pathlib import Path

REPO_DIR = Path('/content/offlinemedia')
BRANCH = 'claude/scan-repo-chatgpt-review-6nljgh'

if REPO_DIR.exists():
    subprocess.run(['git', 'pull', '--rebase', 'origin', BRANCH], cwd=REPO_DIR, check=True)
    print('Repo updated.')
else:
    subprocess.run(
        ['git', 'clone', '--branch', BRANCH,
         'https://github.com/CAption11/offlinemedia', str(REPO_DIR)],
        check=True,
    )
    print('Repo cloned.')

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print('Ready.')

In [ ]:
# Cell 4 — Bootstrap Wan2GP (install + start server)
from portable.wan2gp_bootstrap import bootstrap

process = bootstrap(
    use_google_drive=USE_GOOGLE_DRIVE,
    drive_path=DRIVE_PATH,
    share=True,   # prints a public Gradio link you can open in a browser
)
# process is None when an existing healthy server was reused

In [ ]:
# Cell 5 — Inspect Wan2GP API (run this once to see available endpoints)
# This tells us exactly which API names this version of Wan2GP exposes.
# You will see something like: Parameters - fn_index: 0 / api_name: /generate
# The long_video_generator picks the right one automatically.
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gradio_client'], check=True)

from gradio_client import Client
c = Client('http://127.0.0.1:7860', verbose=False)
c.view_api()

In [ ]:
# Cell 6 — Define your video
# Each prompt = one scene clip (~8 seconds by default).
# 112 prompts = ~15 minutes of video.
# Continuity mode: last frame of each clip seeds the next one.

TITLE = "My Film"

PROMPTS = [
    "A sunrise over misty mountains, golden light piercing through clouds, cinematic wide shot",
    "A hawk soaring through a valley, aerial view, morning light",
    "A river rushing through an ancient forest, slow motion, mist rising",
    # -- add more prompts here --
]

SCENE_DURATION = 8.0   # seconds per clip
FPS = 8                # frames per second (8 is safe on free T4)
WIDTH = 480
HEIGHT = 272           # 16:9 at 480p
USE_CONTINUITY = True  # last frame -> start image of next clip

from pathlib import Path
OUTPUT_DIR = Path('/content/Wan2GP-data/outputs')

n = len(PROMPTS)
est_min = n * SCENE_DURATION / 60
est_hrs = n * 5 / 60
print(f"Plan: {n} scenes, ~{est_min:.1f} min of video, ~{est_hrs:.1f} hrs to generate on T4")

In [ ]:
# Cell 7 — Generate all clips + stitch into final video
# resume=True: if the session crashes, re-run this cell to skip already-done clips.
from portable.long_video_generator import LongVideoGenerator, plan_from_prompts

plan = plan_from_prompts(
    title=TITLE,
    prompts=PROMPTS,
    scene_duration=SCENE_DURATION,
    fps=FPS,
    width=WIDTH,
    height=HEIGHT,
    use_continuity=USE_CONTINUITY,
    output_dir=OUTPUT_DIR,
)

generator = LongVideoGenerator(plan)
final_video = generator.generate(resume=True)
print("\nFinal video saved to:", final_video)

In [ ]:
# Cell 8 — Preview in Colab + download link
from IPython.display import Video, display, FileLink
from pathlib import Path

if not Path(str(final_video)).exists():
    print("Video not found. Run Cell 7 first.")
else:
    size_mb = Path(str(final_video)).stat().st_size / (1024 * 1024)
    print(f"File: {final_video}  ({size_mb:.1f} MB)")
    display(Video(str(final_video), embed=True, width=720))
    display(FileLink(str(final_video)))

In [ ]:
# Cell 9 — (Optional) Copy to Google Drive
# Only needed when USE_GOOGLE_DRIVE = False in Cell 2.
import shutil
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive', force_remount=False)
dest = Path('/content/drive/MyDrive/OfflineMedia-Videos') / Path(str(final_video)).name
dest.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(str(final_video), str(dest))
print("Saved to Google Drive:", dest)